# Batched VIBES on a Colab GPU (benchmark, 3 Nov 2026)

Runs `scripts/vibes_batch.py` / `scripts/vibes_jax.py` (batched ELBO by damped Newton, vectorised L-BFGS or JAX-compiled L-BFGS, + batched interior-point MAP) on the frozen
**19 Oct network VIBES inputs** (4,452 SPARTAN spectra, 2,722 channels, 87-blank rank-70 background) and compares
with that run's saved sigma / ELBO / baselines. Benchmark only: no saved correction is changed.

What each number means:

* **MAP given saved sigma**: the GPU MAP solver alone, fed the saved sigma. Should match the saved baselines to ~1e-6.
* **end to end**: ELBO + MAP. sigma will *not* match the saved run on a few % of spectra, because SciPy's
  relative-reduction stop is path dependent (the fused objective alone differs by >10 % on ~5 % of rows).
* **vs polished**: ELBO and sigma against the 5-restart polished SciPy run (bundle v2 only).
* **cpu_fused**: the adapter's own SciPy procedure (fused objective) on this VM's CPUs, as the same-machine comparator.

Input: `bundle_v2.tar.gz` built on the Mac with
`python research/ftir_hips_chem/workflows/benchmark_vibes_batch.py prepare --out .../bundle_v2 --polished .../cpu_all_polish5.csv`. Put it in Drive at
`My Drive/Aethmodular VIBES/gpu_bench/bundle_v2.tar.gz`, or upload it to `/content/`. Runtime: GPU (G4, L4, A100 or H100).

Results and interpretation: `docs/vibes-batch-gpu-tests-2026-11-03.md` (L-BFGS paths, polish) and
`docs/vibes-elbo-newton-jax-2026-11-03.md` (Newton and JAX).

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
import importlib, subprocess, sys
try:
    import cupy
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x'], check=True)
    import cupy
import cupyx.scipy.special
print('cupy', cupy.__version__, 'devices', cupy.cuda.runtime.getDeviceCount())
x = cupy.random.standard_normal((2048, 2048)); t = cupy.cuda.Event(); t2 = cupy.cuda.Event()
t.record(); [x @ x for _ in range(5)]; t2.record(); t2.synchronize()
print('fp64 GEMM TFLOP/s ~ %.2f' % (5 * 2 * 2048**3 / cupy.cuda.get_elapsed_time(t, t2) / 1e9))
import jax; print('jax', jax.__version__, jax.devices())


In [ ]:
import os, tarfile, json, hashlib, pathlib
DRIVE_DIR = '/content/drive/MyDrive/Aethmodular VIBES/gpu_bench'
src = '/content/bundle_v2.tar.gz'
if not os.path.exists(src):
    from google.colab import drive
    drive.mount('/content/drive')
    src = os.path.join(DRIVE_DIR, 'bundle_v2.tar.gz')
tarfile.open(src).extractall('/content/vb')
B = pathlib.Path('/content/vb/bundle_v2')
man = json.loads((B / 'MANIFEST.json').read_text())
def sha(p):
    h = hashlib.sha256(); h.update(p.read_bytes()); return h.hexdigest()
bad = [n for n, h in man['files'].items() if n != 'MANIFEST.json' and sha(next(B.rglob(n))) != h]
print('bundle rows', man['rows'], 'hash mismatches', bad)
assert not bad

## 0. Damped Newton ELBO (reaches the optimum; ~1.5–4 s for the network on a G4/A100)

Not the upstream SciPy procedure: it reaches the ELBO optimum, i.e. what polish restarts approach. Use `--hess-dtype float32` on a G4
(weak FP64); on an A100 float64 is as fast. Also prints `vs polished`.

In [ ]:
!cd /content/vb && python bundle_v2/code/benchmark_vibes_batch.py run --bundle bundle_v2 --out run_newton32 \
    --backend cupy --rows all --elbo newton --newton-chunk 2048 --hess-dtype float32

In [ ]:
s = json.load(open('/content/vb/run_newton32/summary.json'))
print(json.dumps({k: s[k] for k in ('end_to_end', 'vs_polished') if k in s}, indent=1)[:5000])

## 0b. Latency per optimiser iteration: JAX-compiled L-BFGS vs CuPy L-BFGS vs Newton

In [ ]:
!cd /content/vb && python bundle_v2/code/benchmark_vibes_batch.py latency --bundle bundle_v2 --out latency_v2.json \
    --backend cupy --paths jax,newton,batch --pools 64,512,2048

## 1. Parity pass (96 rows, includes short-stop and slow-tail spectra) + same-VM CPU comparator

In [ ]:
!cd /content/vb && python bundle_v2/code/benchmark_vibes_batch.py run --bundle bundle_v2 --out run_sample96 \
    --backend cupy --rows sample:96 --pool 96 --cpu-ref 96

In [ ]:
s = json.load(open('/content/vb/run_sample96/summary.json'))
print(json.dumps({k: s[k] for k in ('map_given_saved_sigma', 'end_to_end', 'cpu_fused') if k in s}, indent=1)[:6000])

## 2. Recommended: full run with fused SciPy ELBO on every CPU core + GPU MAP

Measured 3 Nov on a G4 VM (48 CPUs): 641 fits/min vs 112 for the current adapter on the same 440 spectra;
the whole network in ~8 min. `--polish 0` is the reference procedure; `--polish 5` fixes SciPy's premature stops
(changes ~5 % of corrections, see docs/vibes-batch-gpu-tests-2026-11-03.md).

In [ ]:
!cd /content/vb && python bundle_v2/code/benchmark_vibes_batch.py run --bundle bundle_v2 --out run_all_cpuelbo \
    --elbo cpu --backend cupy --rows all --cpu-workers $(( $(nproc) - 4 ))

## 3. Comparison: fully batched GPU ELBO (vectorised L-BFGS). Host bound: ~315 fits/min on an A100, ties 48 cores on a G4

In [ ]:
!cd /content/vb && python bundle_v2/code/benchmark_vibes_batch.py run --bundle bundle_v2 --out run_all \
    --backend cupy --rows all --pool 2048 --map-chunk 512

## 4. Optional: same, with up to 5 polish restarts (not the reference procedure; measures the early-stop effect)

In [ ]:
!cd /content/vb && python bundle_v2/code/benchmark_vibes_batch.py run --bundle bundle_v2 --out run_all_polish5 \
    --backend cupy --rows all --pool 2048 --polish 5

## 4b. Optional: JAX-compiled L-BFGS (same optimiser as section 3, compiled; `--polish 5` optional)

In [ ]:
!cd /content/vb && python bundle_v2/code/benchmark_vibes_batch.py run --bundle bundle_v2 --out run_jax \
    --backend cupy --rows all --elbo jax --pool 2048 --chunk-iters 100

## 5. Save results to Drive

In [ ]:
import shutil, time
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.split()[-1]
dst = os.path.join(DRIVE_DIR, 'results_' + time.strftime('%Y%m%d_%H%M') + '_' + gpu)
os.makedirs(dst, exist_ok=True)
for run in ('run_newton32', 'run_sample96', 'run_all_cpuelbo', 'run_all', 'run_all_polish5', 'run_jax'):
    if os.path.exists(f'/content/vb/{run}'):
        shutil.copytree(f'/content/vb/{run}', os.path.join(dst, run), dirs_exist_ok=True)
for f in ('latency_v2.json',):
    if os.path.exists(f'/content/vb/{f}'): shutil.copy2(f'/content/vb/{f}', dst)
print(dst)